# FeedlyShortStories — Google Colab Control Center

Use this notebook as your control panel for:
1. Syncing your GitHub autoblog repository & mounting Google Drive for full backup storage
2. Importing your 2016–present Blogger `.xml` backup into SQLite (`blog.db` + full `archive.db`)
3. Searching your 30,000+ archived posts and auditing duplicate labels
4. Previewing RSS feeds (`--dry-run`) or publishing a custom post directly
5. Triggering your GitHub Actions **webhook** (`repository_dispatch`) on demand

## 1. Mount Google Drive & Clone / Pull Your GitHub Repo
Store your GitHub Personal Access Token (`GH_TOKEN`) and Blogger secrets in **Colab Secrets** (the key icon in the left sidebar).

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
ARCHIVE_DIR = '/content/drive/MyDrive/feedlyshortstories'
os.makedirs(ARCHIVE_DIR, exist_ok=True)
print('Google Drive archive directory:', ARCHIVE_DIR)

In [ ]:
# Replace with your GitHub username and repo name
GITHUB_USER = 'YOUR_GITHUB_USERNAME'
GITHUB_REPO = 'feedlyshortstories'

from google.colab import userdata
import subprocess

try:
    GH_TOKEN = userdata.get('GH_TOKEN')
except Exception:
    GH_TOKEN = ''

repo_dir = f'/content/{GITHUB_REPO}'
if GH_TOKEN:
    remote_url = f'https://{GH_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git'
else:
    remote_url = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git'

if not os.path.exists(repo_dir):
    !git clone {remote_url} {repo_dir}
else:
    %cd {repo_dir}
    !git pull

%cd {repo_dir}
!pip install -q -r requirements.txt

## 2. Import Your Historical Blogger Backup (XML or Google Takeout) into SQLite
Upload either your Blogger `.xml` export (`blog-backup.xml`) **or** your Google Takeout `feed.atom` to `MyDrive/feedlyshortstories/`, and run this cell once.
- `data/blog.db` stays compact for GitHub Actions deduplication.
- `archive.db` in Google Drive stores the **complete HTML bodies** of all archived posts.

In [ ]:
BACKUP_FILE = f'{ARCHIVE_DIR}/feed.atom' if os.path.exists(f'{ARCHIVE_DIR}/feed.atom') else f'{ARCHIVE_DIR}/blog-backup.xml'
FULL_ARCHIVE_DB = f'{ARCHIVE_DIR}/archive.db'

if os.path.exists(BACKUP_FILE):
    !python import_backup.py "{BACKUP_FILE}" --db data/blog.db --archive-db "{FULL_ARCHIVE_DB}" --label-report
else:
    print(f'Upload your Google Takeout feed.atom or Blogger blog-backup.xml to: {ARCHIVE_DIR}')

## 3. Search Your Archive & Check Stats

In [ ]:
!python autoblog.py --stats

In [ ]:
import db, json

QUERY = 'Short Stories'  # Change keyword to search
conn = db.get_connection('data/blog.db')
matches = db.search_articles(conn, QUERY, limit=10)
conn.close()

for m in matches:
    print(f"[{m['published_at'][:10]}] {m['title']}")
    print(f"   Labels: {m['labels_json']} | URL: {m['blogger_url'] or m['source_url']}")

## 4. Test RSS Feeds Safely (`--dry-run`)

In [ ]:
!python autoblog.py --dry-run

## 5. Publish Directly from Colab or Trigger GitHub Actions Webhook
Store `BLOGGER_CLIENT_ID`, `BLOGGER_CLIENT_SECRET`, `BLOGGER_REFRESH_TOKEN`, and `BLOGGER_BLOG_ID` in Colab Secrets if you want to publish directly from Colab, OR just call the GitHub webhook using `GH_TOKEN`.

In [ ]:
import requests

def trigger_github_webhook(event_type='run-rss', payload=None):
    assert GH_TOKEN, 'Set GH_TOKEN in Colab Secrets (repo + workflow scope).'
    url = f'https://api.github.com/repos/{GITHUB_USER}/{GITHUB_REPO}/dispatches'
    resp = requests.post(
        url,
        headers={
            'Authorization': f'Bearer {GH_TOKEN}',
            'Accept': 'application/vnd.github+json',
        },
        json={'event_type': event_type, 'client_payload': payload or {}},
        timeout=15,
    )
    print('GitHub Webhook Status:', resp.status_code, resp.text)

# Example 1: Trigger immediate RSS run on GitHub Actions
# trigger_github_webhook('run-rss', {'dry_run': False})

# Example 2: Publish a custom short story via Webhook
# trigger_github_webhook('publish-post', {
#     'title': 'Midnight Train — A 50-Word Story',
#     'content': '<p>She checked her watch at 11:59 PM...</p>',
#     'labels': ['50-Word Stories', 'Short Stories'],
#     'is_draft': True
# })

## 6. Push Updated `data/blog.db` Back to GitHub (After Importing Backup)

In [ ]:
!git config user.name "Colab Control"
!git config user.email "colab@users.noreply.github.com"
!git add data/blog.db
!git commit -m "Import historical Blogger archive into SQLite" || echo "Nothing to commit"
!git push